# PlagIA Empresarial · 04 — Registro en MLflow, Exportación ONNX (Servidor + Móvil INT8) y Mapas de Calor

**Subsistema de Despliegue MLOps, Edge-AI y Análisis Geoespacial (`ml_engine`)**

### Objetivos Técnicos
1. Consultar y comparar los experimentos en **MLflow** priorizando **F1-Macro y Sensibilidad Fitosanitaria (Recall)**.
2. Registrar el modelo ganador en el **MLflow Model Registry** con alias `@staging`.
3. Exportar el modelo a **ONNX FP32 (`models/plagia_server_fp32.onnx`)** y cuantizarlo a **INT8 (`models/plagia_mobile_int8.onnx`)** para **Android Studio / Flutter**.
4. Ejecutar el benchmark de latencia **PyTorch vs. ONNX Runtime** y generar el **Mapa de Calor Fitosanitario GeoJSON y PNG** con **GeoPandas**.

In [ ]:
import sys
from pathlib import Path

RAIZ_PROYECTO = Path.cwd().resolve()
while not (RAIZ_PROYECTO / "pyproject.toml").exists() and RAIZ_PROYECTO != RAIZ_PROYECTO.parent:
    RAIZ_PROYECTO = RAIZ_PROYECTO.parent
if str(RAIZ_PROYECTO) not in sys.path:
    sys.path.insert(0, str(RAIZ_PROYECTO))

import mlflow
from mlflow.tracking import MlflowClient

from ml_engine.architectures import construir_clasificador_pytorch
from ml_engine.config import (
    CLASES_DRON,
    MLFLOW_EXPERIMENT_NAME,
    MLFLOW_MODEL_NAME,
    MLFLOW_TRACKING_URI,
)
from ml_engine.exporter import (
    asegurar_artefactos_por_defecto,
    comparar_latencia_pytorch_vs_onnx,
)
from ml_engine.geospatial import (
    construir_geodataframe_mapa_calor,
    exportar_artefactos_mapa_calor,
)

## 1. Selección de la Mejor Corrida y Promoción en MLflow Model Registry

In [ ]:
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow.set_experiment(MLFLOW_EXPERIMENT_NAME)

df_corridas = mlflow.search_runs(order_by=["metrics.f1_macro DESC"])
if not df_corridas.empty:
    mejor_corrida = df_corridas.iloc[0]
    print("Mejor corrida seleccionada:", mejor_corrida.get("tags.mlflow.runName"))
    print("F1-Macro :", mejor_corrida.get("metrics.f1_macro"))
    print("Recall   :", mejor_corrida.get("metrics.recall_macro"))
    uri_modelo = f"runs:/{mejor_corrida['run_id']}/model"
    reg = mlflow.register_model(model_uri=uri_modelo, name=MLFLOW_MODEL_NAME)
    cliente = MlflowClient()
    cliente.set_registered_model_alias(MLFLOW_MODEL_NAME, "staging", reg.version)
    print(f"Modelo '{MLFLOW_MODEL_NAME}' v{reg.version} promovido al alias '@staging'.")
else:
    print("No hay corridas previas en MLflow; se exportará el artefacto base para validación.")

## 2. Exportación a ONNX (Servidor + Móvil INT8 para Flutter) y Benchmark

In [ ]:
meta_export = asegurar_artefactos_por_defecto(class_names=CLASES_DRON)
print("Metadatos de exportación ONNX:")
print(" - Servidor ONNX (MB):", meta_export["server_size_mb"])
print(" - Móvil INT8 (MB)   :", meta_export["mobile_size_mb"])

modelo_ref = construir_clasificador_pytorch("efficientnet_b0", num_classes=len(CLASES_DRON), pretrained=False)
bench = comparar_latencia_pytorch_vs_onnx(modelo_ref)
print("\nBenchmark de Inferencia:")
for k, v in bench.items():
    print(f" - {k}: {v}")

## 3. Generación del Mapa de Calor Fitosanitario con GeoPandas (GeoJSON + PNG)

In [ ]:
detecciones_muestra = [
    {"patch_id": "P01", "disease_label": "sano", "confidence": 0.96, "bbox_gps": (-74.895, 4.142, -74.890, 4.147)},
    {"patch_id": "P02", "disease_label": "roya_parda", "confidence": 0.91, "bbox_gps": (-74.890, 4.142, -74.885, 4.147)},
    {"patch_id": "P03", "disease_label": "roya_amarilla", "confidence": 0.89, "bbox_gps": (-74.885, 4.142, -74.880, 4.147)},
    {"patch_id": "P04", "disease_label": "sano", "confidence": 0.94, "bbox_gps": (-74.895, 4.147, -74.890, 4.152)},
    {"patch_id": "P05", "disease_label": "roya_parda", "confidence": 0.95, "bbox_gps": (-74.890, 4.147, -74.885, 4.152)},
    {"patch_id": "P06", "disease_label": "sano", "confidence": 0.92, "bbox_gps": (-74.885, 4.147, -74.880, 4.152)},
]

gdf = construir_geodataframe_mapa_calor(detecciones_muestra, lot_id="LOTE-ESPINAL-01", municipio="El Espinal")
info_mapa = exportar_artefactos_mapa_calor(gdf, lot_id="LOTE-ESPINAL-01")
print("Mapa de calor exportado:")
print(" - GeoJSON :", info_mapa["geojson_path"])
print(" - PNG     :", info_mapa["png_path"])
print(" - Tasa de infestación del lote:", f"{info_mapa['infection_rate']*100:.1f}%")